# 00 - Capability check

Run once (interactively) on the Free Edition workspace to confirm what the pipeline can rely on:

| Check | Why it matters |
|---|---|
| Spark session | Every pipeline task needs it |
| Landing volume readable | Source for Auto Loader |
| PyPI package `langdetect` | Language detection in `03_enrich` (stage 1) |
| Serving endpoints (`embedding_endpoint`, `role_llm_endpoint`, `llm_endpoint`) | Foundation Model endpoints `03_enrich` calls; same widget names and defaults |
| `ai_query` embeddings | Role matching in `03_enrich` (stage 2) |
| `ai_query` structured JSON (`responseFormat`) | Role tie-break, custom role check and skills LLM in `03_enrich` |

Nothing is written by this notebook.

## Before running: add dependencies in the Environment panel (do NOT use `%pip`)

On serverless, `%pip install` builds a notebook environment that Spark must download; on Free Edition this can fail with `ENVIRONMENT_DOWNLOAD_USER_ERROR.NOT_FOUND`.

1. Open the notebook, click the **Environment** icon in the right side panel.
2. Under **Dependencies** add one line: `langdetect==1.0.9`.
3. Click **Apply** (the session restarts), then **Run all**.

If Spark still fails, click **Connect → Reset session** (or detach/reattach) and run again.

In [ ]:
dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en", "Embedding endpoint")
dbutils.widgets.text("role_llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "Role tie-break LLM endpoint")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct", "Skills LLM endpoint")

LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint").strip()
ROLE_LLM_ENDPOINT = dbutils.widgets.get("role_llm_endpoint").strip()
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
ENDPOINTS = {"embedding_endpoint": EMBEDDING_ENDPOINT, "role_llm_endpoint": ROLE_LLM_ENDPOINT, "llm_endpoint": LLM_ENDPOINT}

results = {}


def record(check, ok, detail=""):
    results[check] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)[:300]}
    print(f"[{'PASS' if ok else 'FAIL'}] {check}: {detail}")

## Environment

In [ ]:
try:
    current = spark.sql("SELECT current_catalog() AS catalog, current_user() AS user").first()
    record("spark_session", True, f"catalog={current.catalog}, user={current.user}, spark={spark.version}")
except Exception as exc:
    message = str(exc)
    hint = " -> remove %pip, add deps in the Environment panel, then Reset session" if "ENVIRONMENT_DOWNLOAD" in message else ""
    record("spark_session", False, message.splitlines()[0] + hint)

## Landing volume

In [ ]:
try:
    folders = dbutils.fs.ls(LANDING_PATH)
    csv_files = []
    for folder in folders:
        if folder.isDir():
            csv_files.extend(f for f in dbutils.fs.ls(folder.path) if f.name.endswith(".csv"))
        elif folder.name.endswith(".csv"):
            csv_files.append(folder)

    by_source = {}
    for file_info in csv_files:
        prefix = file_info.name.split("_", 1)[0]
        by_source[prefix] = by_source.get(prefix, 0) + 1

    record("landing_volume", True, f"{len(folders)} entries, {len(csv_files)} csv files, by source={by_source}")
except Exception as exc:
    record("landing_volume", False, exc)

## Python packages (PyPI)

In [ ]:
try:
    from langdetect import DetectorFactory, detect_langs

    DetectorFactory.seed = 0
    samples = {
        "en": "We are looking for a senior data engineer with PySpark experience.",
        "fr": "Nous recherchons un analyste marketing pour rejoindre notre équipe à Paris.",
        "es": "Buscamos un perfil proactivo con experiencia en medios y manejo de Excel.",
    }
    detected = {expected: str(detect_langs(text)[0]) for expected, text in samples.items()}
    record("langdetect", True, detected)
except Exception as exc:
    record("langdetect", False, exc)

## Serving endpoints

In [ ]:
try:
    from databricks.sdk import WorkspaceClient

    available = {endpoint.name for endpoint in WorkspaceClient().serving_endpoints.list()}
    for widget, name in ENDPOINTS.items():
        found = name in available
        record(f"endpoint_{widget}", found, name if found else f"{name} not found, available: {sorted(available)[:15]}")
except Exception as exc:
    record("serving_endpoints", False, exc)

## `ai_query` (embeddings + structured JSON)

In [ ]:
import json

import numpy as np

try:
    texts = ["Senior Data Engineer PySpark Databricks", "Data Engineering pipelines ETL", "Payroll HR operations"]
    columns = ", ".join(f"CAST(ai_query('{EMBEDDING_ENDPOINT}', '{text}') AS ARRAY<FLOAT>) AS e{i}" for i, text in enumerate(texts))
    row = spark.sql(f"SELECT {columns}").first()
    vectors = np.array([row[f"e{i}"] for i in range(len(texts))], dtype=np.float32)
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    similar = float(vectors[0] @ vectors[1])
    dissimilar = float(vectors[0] @ vectors[2])
    record("ai_query_embedding", similar > dissimilar,
           f"{EMBEDDING_ENDPOINT}: dim={vectors.shape[1]}, similar={similar:.3f}, dissimilar={dissimilar:.3f}")
except Exception as exc:
    record("ai_query_embedding", False, f"{EMBEDDING_ENDPOINT}: {exc}")

# Same responseFormat style as 03_enrich (json_schema, strict, enum)
CHECK_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "capability_check",
        "schema": {
            "type": "object",
            "properties": {"answer": {"type": "string", "enum": ["OK", "NO"]}},
            "required": ["answer"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})
for widget in ("role_llm_endpoint", "llm_endpoint"):
    endpoint = ENDPOINTS[widget]
    try:
        reply = spark.sql(
            f"SELECT ai_query('{endpoint}', 'Return JSON with answer set to OK', responseFormat => '{CHECK_FORMAT}') AS reply"
        ).first().reply
        record(f"ai_query_{widget}", json.loads(reply).get("answer") == "OK", f"{endpoint}: {reply!r}")
    except Exception as exc:
        record(f"ai_query_{widget}", False, f"{endpoint}: {exc}")

## Summary

In [ ]:
# Plain-text summary first so it prints even when the Spark session is broken
width = max(len(check) for check in results)
for check, value in results.items():
    print(f"{value['status']:<5} {check:<{width}}  {value['detail']}")

try:
    summary = [(check, value["status"], value["detail"]) for check, value in results.items()]
    display(spark.createDataFrame(summary, "check STRING, status STRING, detail STRING"))
except Exception as exc:
    print(f"(table display skipped, Spark unavailable: {str(exc).splitlines()[0]})")